# KNN do zero — busca ampla de configurações

Usamos o mesmo conjunto padronizado dos notebooks anteriores. A referência é o KNN da primeira submissão, com K igual a 11, distância Manhattan, escala por quartis, voto simples e peso dobrado para glicose e hemoglobina.

O objetivo aqui é descobrir quais mudanças se sustentam quando trocamos as divisões. Começamos com uma grade de configurações definida no código, fazemos a escolha dentro das divisões de treinamento e comparamos as previsões em amostras que ficaram fora de cada escolha.

O KNN e as métricas foram implementados no notebook. NumPy é usado para calcular muitas distâncias de uma vez. Para rodar no VS Code, o kernel precisa ter NumPy. A versão de Python e NumPy usada nesta execução aparece na primeira saída.

### Resultado da busca

| Avaliação | Primeira submissão | Busca ampla |
|---|---:|---:|
| F1 macro médio nas dez rodadas externas | 86,63% | 86,65% |
| Acurácia média nas dez rodadas externas | 86,66% | 86,66% |
| F1 macro nas 104 amostras reservadas | 86,53% | 84,52% |
| Acertos nas 104 amostras reservadas | 90 | 88 |

A configuração que ficou em primeiro **dentro da busca** chegou a 90,39% de acurácia média. Ela usa Manhattan, K igual a 19, voto simples e pelo menos 9 votos para diabetes (45% arredondado para o próximo voto inteiro). Usa os nove atributos numéricos, peso 2 para glicose e 3 para hemoglobina. A glicose ausente fica fora da distância. A escala usa a mediana e o desvio absoluto mediano, que é a mediana das distâncias dos valores até a mediana da coluna, multiplicada por 1,4826.

Nas 104 amostras reservadas, a candidata acertou 48 casos de diabetes contra 46 da primeira submissão. Em controle, acertou 40 contra 44. A busca encontrou versões com outro equilíbrio entre as classes, mas o resultado nas rodadas externas ficou praticamente igual e a candidata perdeu dois acertos na reserva. Mantemos a primeira submissão como referência.


### Dados e divisão

Reproduzimos a divisão estratificada com semente 42 do notebook anterior. Das 520 amostras, usamos 416 na busca e guardamos as mesmas 104 para a comparação local ao final. Essas 104 já foram consultadas no trabalho anterior, portanto são uma referência histórica.

Conferimos que cada paciente aparece uma única vez na base padronizada. A coluna class só entra na divisão estratificada, no voto e no cálculo das métricas. O teste da competição fica fora deste notebook.

In [1]:
import csv
import hashlib
import json
import random
import sys
from dataclasses import asdict, dataclass, replace
from itertools import product
from pathlib import Path
from time import perf_counter

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho_base = raiz / "dados" / "treinamento_padronizado.csv"
with caminho_base.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("A divisão exige um registro por paciente na base padronizada.")

rotulos = np.array([int(a["class"]) for a in amostras], dtype=np.int8)
controle = np.flatnonzero(rotulos == 0).tolist()
diabetes = np.flatnonzero(rotulos == 1).tolist()
if len(controle) + len(diabetes) != len(amostras):
    raise ValueError("Esperávamos apenas os rótulos 0 e 1.")

gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0 = round(0.20 * len(controle))
n1 = round(0.20 * len(diabetes))
reservadas = np.array(controle[:n0] + diabetes[:n1])
desenvolvimento = np.array(controle[n0:] + diabetes[n1:])


def dividir(indices, quantidade, semente):
    partes = [[] for _ in range(quantidade)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if rotulos[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            partes[posicao % quantidade].append(i)

    for posicao, parte in enumerate(partes):
        ajuste = [i for j, outra in enumerate(partes) if j != posicao for i in outra]
        yield np.array(ajuste), np.array(parte)


print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")
print("SHA-256 da base:", hashlib.sha256(caminho_base.read_bytes()).hexdigest())
print("Busca:", len(desenvolvimento), "| Reservadas:", len(reservadas))

Python 3.12.14 | NumPy 2.3.5
SHA-256 da base: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032
Busca: 416 | Reservadas: 104


### Atributos e preparação

A base numérica tem as nove medidas que já usávamos. Testamos também retirar medidas, usar grupos menores e criar IMC ou pressão de pulso. A pressão de pulso é a diferença entre as pressões sistólica e diastólica. Esses atributos derivados são calculados por amostra.

As escalas, os limites de corte, as medianas de preenchimento e as categorias conhecidas são calculados com a parte de referência de cada rodada. Mantemos a regra anterior para a hemoglobina acima de 100, sob a premissa de HbA1c. Também testamos o logaritmo de algumas medidas e o corte dos extremos pelos percentis 1 e 99.

Para valores ausentes, comparamos preencher pela mediana, deixar uma medida fora da distância e acrescentar indicadores de ausência dos exames. Um indicador vale 1 quando o exame falta e 0 quando está disponível. As categorias ausentes ou inéditas ficam fora da comparação categórica.

In [2]:
numericas = (
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC", "GLUCOSE",
    "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
)
nomes_num = numericas + ("IMC", "PULSE_PRESSURE", "FALTA_GLUCOSE", "FALTA_HEMOGLOBIN")
indice_num = {nome: i for i, nome in enumerate(nomes_num)}
exames = ("GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM")
nomes_cat = (
    "SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY", "GENDER",
    "BLOOD_GROUP", "RH_FACTOR", "COLLECTION_SITE", "LAB_DEVICE",
)


def ler_medida(amostra, coluna):
    texto = amostra[coluna].strip().replace(",", ".")
    try:
        valor = float(texto)
    except ValueError:
        return np.nan
    if not np.isfinite(valor) or valor <= 0:
        return np.nan
    if coluna == "HEMOGLOBIN" and valor > 100:
        return np.nan
    return valor


def ler_categoria(amostra, coluna):
    valor = amostra[coluna].strip()
    if valor.lower() in {"", "desconhecido"} or (coluna == "GENDER" and valor == "N"):
        return ""
    return valor


medidas = np.array([[ler_medida(a, c) for c in numericas] for a in amostras])
altura_m = medidas[:, indice_num["HEIGHT"]] / 100
imc = medidas[:, indice_num["WEIGHT"]] / altura_m ** 2
pulso = medidas[:, indice_num["SYSTOLIC"]] - medidas[:, indice_num["DIASTOLIC"]]
valores = np.column_stack((
    medidas, imc, pulso,
    np.isnan(medidas[:, indice_num["GLUCOSE"]]),
    np.isnan(medidas[:, indice_num["HEMOGLOBIN"]]),
))
categorias = {c: np.array([ler_categoria(a, c) for a in amostras]) for c in nomes_cat}


def preparar_medidas(ajuste, consulta, escala, ausencia, transformacao):
    referencia = valores[ajuste].copy()
    avaliacao = valores[consulta].copy()

    for j, nome in enumerate(nomes_num):
        if nome.startswith("FALTA_"):
            continue

        ref = referencia[:, j]
        aval = avaliacao[:, j]
        presentes = ref[np.isfinite(ref)]
        if not len(presentes):
            raise ValueError(f"Nenhuma medida disponível na referência para {nome}.")

        if transformacao == "log" and nome in exames:
            presentes = np.log1p(presentes)
            ref[:] = np.log1p(ref)
            aval[:] = np.log1p(aval)
        if transformacao == "corte":
            minimo, maximo = np.percentile(presentes, [1, 99])
            presentes = np.clip(presentes, minimo, maximo)
            ref[:] = np.clip(ref, minimo, maximo)
            aval[:] = np.clip(aval, minimo, maximo)

        mediana = np.median(presentes)
        omitir = (
            ausencia == "todas"
            or (ausencia == "glicose" and nome == "GLUCOSE")
            or (ausencia == "exames" and nome in ("GLUCOSE", "HEMOGLOBIN"))
        )
        if not omitir:
            ref[np.isnan(ref)] = mediana
            aval[np.isnan(aval)] = mediana

        if escala == "percentis":
            unicos, contagens = np.unique(presentes, return_counts=True)
            postos = (np.cumsum(contagens) - contagens / 2) / len(presentes)
            ref[:] = np.interp(ref, unicos, postos)
            aval[:] = np.interp(aval, unicos, postos)
            continue

        centro = mediana
        if escala == "quartis":
            q1, q3 = np.percentile(presentes, [25, 75])
            divisor = q3 - q1
        elif escala == "desvio":
            centro = np.mean(presentes)
            divisor = np.std(presentes, ddof=1) if len(presentes) > 1 else 0
        elif escala == "mad":
            divisor = 1.4826 * np.median(np.abs(presentes - mediana))
        else:
            centro = np.min(presentes)
            divisor = np.ptp(presentes)

        if divisor == 0:
            divisor = np.ptp(presentes) or 1.0
        ref[:] = (ref - centro) / divisor
        aval[:] = (aval - centro) / divisor

    return referencia, avaliacao

### Configurações que vamos comparar

A grade tem dois níveis. Primeiro, variamos uma família de escolhas por vez em relação à submissão. Depois, cruzamos um conjunto menor de escalas, distâncias, pesos, ausências e pesos categóricos para detectar interações. Cada configuração de distância é combinada com os mesmos valores de K, regras de voto e limites para prever diabetes.

As distâncias numéricas incluem Manhattan (média das diferenças absolutas), Euclidiana (raiz da média dos quadrados), Chebyshev ponderada (maior diferença após o peso) e Canberra (diferenças relativas à magnitude dos dois valores). As três primeiras passam por D / (1 + D) antes de combinar com as categorias. Canberra já fica entre 0 e 1. Para as categorias, comparamos Jaccard das colunas one-hot com a proporção de atributos diferentes.

No voto, testamos voto simples, voto duplo para os três primeiros, peso por posição, peso inverso da distância e peso inverso do quadrado da distância. Quando há vizinhos a distância zero nas duas últimas regras, só eles votam. Os limites de 45%, 50% e 55% usam a proporção de votos de diabetes. No empate exato de 50%, usamos a classe do vizinho mais próximo.

In [3]:
@dataclass(frozen=True)
class Distancia:
    escala: str = "quartis"
    metrica: str = "manhattan"
    ausencia: str = "mediana"
    transformacao: str = "nenhuma"
    medidas: tuple = numericas
    pesos: tuple = (("GLUCOSE", 2), ("HEMOGLOBIN", 2))
    categorias: tuple = ("SMOKER",)
    peso_cat: float = 0.10
    metrica_cat: str = "jaccard"


base = Distancia()
familias = {base: "Referência"}


def incluir(familia, **mudancas):
    familias.setdefault(replace(base, **mudancas), familia)


for escala in ("quartis", "desvio", "mad", "minmax", "percentis"):
    incluir("Escala", escala=escala)
for metrica in ("euclidiana", "chebyshev", "canberra"):
    incluir("Distância", metrica=metrica)
for ausencia in ("glicose", "exames", "todas"):
    incluir("Ausências", ausencia=ausencia)
    incluir("Ausências", ausencia=ausencia, medidas=numericas + nomes_num[-2:])
incluir("Ausências", medidas=numericas + nomes_num[-2:])
for transformacao in ("log", "corte"):
    incluir("Transformação", transformacao=transformacao)

for glicose, hemo in product((1, 2, 3, 4, 6), repeat=2):
    incluir("Pesos dos exames", pesos=(("GLUCOSE", glicose), ("HEMOGLOBIN", hemo)))
for coluna, peso in product(("TRIGLYCERIDES", "SYSTOLIC", "DIASTOLIC", "AGE", "WEIGHT"), (0.5, 2, 3)):
    incluir("Peso de outra medida", pesos=base.pesos + ((coluna, peso),))
for coluna in numericas:
    incluir("Retirar uma medida", medidas=tuple(c for c in numericas if c != coluna))

sem_altura_peso = tuple(c for c in numericas if c not in ("HEIGHT", "WEIGHT"))
sem_pressao = tuple(c for c in numericas if c not in ("SYSTOLIC", "DIASTOLIC"))
for medidas_ativas in (
    exames, exames[:3], exames[:2], ("AGE",) + exames,
    tuple(c for c in numericas if c not in exames),
    numericas + ("IMC",), sem_altura_peso + ("IMC",),
    sem_pressao + ("SYSTOLIC", "PULSE_PRESSURE"),
    sem_altura_peso + ("IMC", "PULSE_PRESSURE"),
):
    incluir("Grupos e medidas derivadas", medidas=medidas_ativas)

pacotes = (
    ("SMOKER",), ("FAMILY_HISTORY",), ("PHYSICAL_ACTIVITY",), ("GENDER",),
    ("BLOOD_GROUP", "RH_FACTOR"), ("SMOKER", "FAMILY_HISTORY"),
    ("SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY"), nomes_cat[:6],
    ("SMOKER", "COLLECTION_SITE"), ("SMOKER", "LAB_DEVICE"),
)
incluir("Categorias", categorias=(), peso_cat=0)
for pacote, peso_cat, metrica_cat in product(pacotes, (0.05, 0.10, 0.25, 0.50), ("jaccard", "hamming")):
    incluir("Categorias", categorias=pacote, peso_cat=peso_cat, metrica_cat=metrica_cat)

for escala, metrica, pesos_dupla, ausencia, peso_cat in product(
    ("quartis", "mad", "percentis"), ("manhattan", "euclidiana"),
    ((1, 1), (2, 2), (2, 3), (2, 4)), ("mediana", "glicose"), (0.0, 0.10)
):
    incluir(
        "Combinações planejadas", escala=escala, metrica=metrica, ausencia=ausencia,
        pesos=tuple(zip(("GLUCOSE", "HEMOGLOBIN"), pesos_dupla)),
        categorias=("SMOKER",) if peso_cat else (), peso_cat=peso_cat
    )

configuracoes = list(familias)
valores_k = (1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 25, 31, 41, 51)
votos = ("simples", "duplo3", "por_posicao", "inverso", "inverso_quadrado")
limites = (0.45, 0.50, 0.55)
regras = list(product(votos, valores_k, limites))
formato_busca = (len(configuracoes), len(regras))
total_configuracoes = len(configuracoes) * len(regras)
indice_base = regras.index(("simples", 11, 0.50))
print(f"{len(configuracoes)} configurações de distância × {len(regras)} regras de voto")
print(f"{total_configuracoes:,} combinações completas")
for nome in dict.fromkeys(familias.values()):
    print(f"{nome}: {sum(f == nome for f in familias.values())}")

241 configurações de distância × 225 regras de voto
54,225 combinações completas
Referência: 1
Escala: 4
Distância: 3
Ausências: 7
Transformação: 2
Pesos dos exames: 24
Peso de outra medida: 15
Retirar uma medida: 9
Grupos e medidas derivadas: 9
Categorias: 80
Combinações planejadas: 87


### Cálculo das distâncias e da votação

Cada linha da matriz representa uma amostra que queremos classificar. Cada coluna representa um possível vizinho. Preparamos os atributos uma vez por escala e regra de ausência dentro da rodada. Para cada configuração, ordenamos os vizinhos uma vez e aproveitamos a ordem para calcular todos os valores de K.

A distância numérica usa os pesos das medidas que puderam ser comparadas. Quando todas as medidas são omitidas, a distância numérica fica em 1. Se nenhuma categoria puder ser comparada, usamos só o bloco numérico. Nos empates de distância, preservamos uma ordem sorteada com a semente 42.

In [4]:
def distancia_categorias(ajuste, consulta, colunas, metrica):
    comparadas = np.zeros((len(consulta), len(ajuste)), dtype=np.int16)
    iguais = np.zeros_like(comparadas)
    for coluna in colunas:
        ref = categorias[coluna][ajuste]
        aval = categorias[coluna][consulta]
        conhecidas = np.unique(ref[ref != ""])
        vale = np.isin(aval, conhecidas)[:, None] & (ref != "")[None, :]
        comparadas += vale
        iguais += vale & (aval[:, None] == ref[None, :])

    divisor = 2 * comparadas - iguais if metrica == "jaccard" else comparadas
    return np.divide(
        divisor - iguais, divisor,
        out=np.full(divisor.shape, np.nan), where=divisor > 0,
    )


def ordenar_configuracao(config, ajuste, consulta, cache_num, cache_cat):
    chave_num = (config.escala, config.ausencia, config.transformacao)
    if chave_num not in cache_num:
        cache_num[chave_num] = preparar_medidas(ajuste, consulta, *chave_num)
    referencia, avaliacao = cache_num[chave_num]
    js = [indice_num[c] for c in config.medidas]
    mapa_pesos = dict(config.pesos)
    pesos = np.array([mapa_pesos.get(c, 1) for c in config.medidas])
    ref = referencia[:, js]
    aval = avaliacao[:, js]
    dif = np.abs(aval[:, None, :] - ref[None, :, :])
    disponiveis = np.isfinite(dif)
    total_pesos = disponiveis @ pesos
    dif = np.nan_to_num(dif, nan=0.0)

    if config.metrica == "canberra":
        magnitudes = np.abs(aval[:, None, :]) + np.abs(ref[None, :, :])
        dif = np.divide(dif, magnitudes, out=np.zeros_like(dif), where=magnitudes > 0)
    elif config.metrica == "euclidiana":
        dif **= 2

    if config.metrica == "chebyshev":
        distancia = np.max(dif * pesos, axis=-1)
    else:
        distancia = np.divide(
            dif @ pesos, total_pesos,
            out=np.zeros(total_pesos.shape), where=total_pesos > 0,
        )
    if config.metrica == "euclidiana":
        distancia = np.sqrt(distancia)
    if config.metrica != "canberra":
        distancia = distancia / (1 + distancia)
    distancia[total_pesos == 0] = 1.0

    if config.peso_cat:
        chave_cat = (config.categorias, config.metrica_cat)
        if chave_cat not in cache_cat:
            cache_cat[chave_cat] = distancia_categorias(ajuste, consulta, *chave_cat)
        cat = cache_cat[chave_cat]
        distancia = np.where(
            np.isnan(cat), distancia,
            (1 - config.peso_cat) * distancia + config.peso_cat * cat,
        )

    ordem = np.argsort(distancia, axis=1, kind="stable")[:, :max(valores_k)]
    return rotulos[ajuste][ordem], np.take_along_axis(distancia, ordem, axis=1)


def previsoes_todas(classes, distancias):
    posicoes = np.arange(1, classes.shape[1] + 1)
    ks = np.array(valores_k) - 1
    previsoes = []
    for voto in votos:
        if voto == "simples":
            pesos = np.ones_like(distancias)
        elif voto == "duplo3":
            pesos = np.broadcast_to(np.where(posicoes <= 3, 2, 1), distancias.shape)
        elif voto == "por_posicao":
            pesos = np.broadcast_to(1 / np.sqrt(posicoes), distancias.shape)
        else:
            expoente = 1 if voto == "inverso" else 2
            zeros = distancias == 0
            pesos = 1 / np.maximum(distancias, 1e-12) ** expoente
            pesos = np.where(zeros[:, :1], zeros, pesos)

        positivos = np.cumsum(pesos * classes, axis=1)[:, ks]
        total = np.cumsum(pesos, axis=1)[:, ks]
        proporcao = positivos / total
        for j in range(len(ks)):
            for limite in limites:
                prevista = proporcao[:, j] >= limite
                if limite == 0.50:
                    empatou = positivos[:, j] * 2 == total[:, j]
                    prevista = np.where(empatou, classes[:, 0], prevista)
                previsoes.append(prevista)

    return np.array(previsoes, dtype=bool)


def contar(reais, previstas):
    verdade = reais.astype(bool)
    return np.stack([
        np.sum(~previstas & ~verdade, axis=-1), np.sum(previstas & ~verdade, axis=-1),
        np.sum(~previstas & verdade, axis=-1), np.sum(previstas & verdade, axis=-1),
    ], axis=-1).astype(np.int16)


def razao(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(contagens, -1, 0)
    f0 = razao(2 * tn, 2 * tn + fp + fn)
    f1 = razao(2 * tp, 2 * tp + fp + fn)
    p0 = razao(tn, tn + fn)
    p1 = razao(tp, tp + fp)
    return {
        "acuracia": razao(tn + tp, tn + fp + fn + tp),
        "f1": (f0 + f1) / 2,
        "precisao": (p0 + p1) / 2,
        "recall0": razao(tn, tn + fp),
        "recall1": razao(tp, tp + fn),
    }


def executar_divisao(ajuste, consulta, indices_config=None):
    ajuste = ajuste.tolist()
    random.Random(42).shuffle(ajuste)
    ajuste = np.array(ajuste)
    if len(ajuste) < max(valores_k):
        raise ValueError("A referência tem menos amostras do que o maior K.")

    cache_num, cache_cat = {}, {}
    if indices_config is None:
        indices_config = range(len(configuracoes))
    resultados = []
    for i in indices_config:
        classes, distancias = ordenar_configuracao(
            configuracoes[i], ajuste, consulta, cache_num, cache_cat
        )
        resultados.append(contar(rotulos[consulta], previsoes_todas(classes, distancias)))
    return np.array(resultados)


def ordenar_busca(rodadas):
    medidas = metricas(rodadas)
    f1 = medidas["f1"].mean(axis=0).ravel()
    acuracia = medidas["acuracia"].mean(axis=0).ravel()
    return np.lexsort((np.arange(f1.size), -acuracia, -f1))


def ler_configuracao(posicao):
    i, j = np.unravel_index(posicao, formato_busca)
    return int(i), int(j)


def descrever(posicao):
    i, j = ler_configuracao(posicao)
    return f"{familias[configuracoes[i]]} | voto={regras[j][0]}, K={regras[j][1]}, limite={regras[j][2]:.0%}"

### Escolha e avaliação em rodadas separadas

Repetimos a avaliação externa com duas sementes. Em cada uma das cinco rodadas de uma repetição, uma parte fica guardada e as outras quatro entram na busca. Dentro dessas quatro partes fazemos outras três divisões, testamos a grade e escolhemos pelo F1 macro médio. Quando há empate, usamos a acurácia e depois a ordem fixa da grade.

Em seguida, a configuração escolhida faz previsões para a parte externa daquela rodada. A versão submetida ao Kaggle é avaliada na mesma parte. As dez pontuações externas mostram como se saíram tanto a busca quanto a referência. Como as duas repetições usam as mesmas pessoas em partições diferentes, as rodadas não são dez amostras independentes.

In [5]:
inicio = perf_counter()
sementes_externas = (42, 73)
internas = []
externas = []
selecionadas = []

for semente in sementes_externas:
    for numero, (ajuste, consulta) in enumerate(dividir(desenvolvimento, 5, semente), start=1):
        rodadas_internas = np.array([
            executar_divisao(ref, aval)
            for ref, aval in dividir(ajuste, 3, semente * 100 + numero)
        ])
        melhor = int(ordenar_busca(rodadas_internas)[0])
        i, j = ler_configuracao(melhor)
        resultados = executar_divisao(ajuste, consulta, [0, i])
        referencia = resultados[0, indice_base]
        candidata = resultados[1, j]
        internas.append(rodadas_internas)
        externas.append([referencia, candidata])
        selecionadas.append(melhor)
        print(
            f"Semente {semente}, rodada {numero} | {descrever(melhor)} | "
            f"F1 ref. {metricas(referencia)['f1']:.2%} | F1 busca {metricas(candidata)['f1']:.2%}"
        )

externas = np.array(externas)
print(f"Tempo das rodadas externas: {(perf_counter() - inicio):.1f} s")

Semente 42, rodada 1 | Distância | voto=inverso, K=5, limite=55% | F1 ref. 94.03% | F1 busca 89.25%
Semente 42, rodada 2 | Combinações planejadas | voto=inverso_quadrado, K=21, limite=45% | F1 ref. 86.72% | F1 busca 86.72%
Semente 42, rodada 3 | Combinações planejadas | voto=duplo3, K=7, limite=50% | F1 ref. 83.13% | F1 busca 83.13%
Semente 42, rodada 4 | Pesos dos exames | voto=inverso, K=11, limite=45% | F1 ref. 86.68% | F1 busca 86.74%
Semente 42, rodada 5 | Combinações planejadas | voto=simples, K=15, limite=45% | F1 ref. 90.36% | F1 busca 87.94%
Semente 73, rodada 1 | Pesos dos exames | voto=duplo3, K=15, limite=45% | F1 ref. 80.94% | F1 busca 82.14%
Semente 73, rodada 2 | Combinações planejadas | voto=simples, K=9, limite=45% | F1 ref. 90.35% | F1 busca 90.36%
Semente 73, rodada 3 | Combinações planejadas | voto=por_posicao, K=17, limite=50% | F1 ref. 85.49% | F1 busca 85.54%
Semente 73, rodada 4 | Combinações planejadas | voto=inverso_quadrado, K=51, limite=45% | F1 ref. 84.33% 

In [6]:
def linha_metricas(nome, contagens):
    m = metricas(contagens)
    medias = [float(np.mean(m[c])) for c in ("acuracia", "f1", "precisao", "recall0", "recall1")]
    print(f"{nome:23} | " + " | ".join(f"{valor:>8.2%}" for valor in medias))


print("Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
linha_metricas("Primeira submissão", externas[:, 0])
linha_metricas("Busca aninhada", externas[:, 1])
f1_externo = metricas(externas)["f1"]
melhoras_externas = np.sum(f1_externo[:, 1] > f1_externo[:, 0])
empates_externos = np.sum(f1_externo[:, 1] == f1_externo[:, 0])
print(f"F1 da busca maior em {melhoras_externas}/10 rodadas; empates: {empates_externos}")
print("Desvio padrão do F1 externo:",
      f"referência {np.std(f1_externo[:, 0], ddof=1):.2%},",
      f"busca {np.std(f1_externo[:, 1], ddof=1):.2%}")

Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Primeira submissão      |   86.66% |   86.63% |   87.28% |   92.75% |   81.00%
Busca aninhada          |   86.66% |   86.65% |   86.78% |   88.75% |   84.72%
F1 da busca maior em 6/10 rodadas; empates: 2
Desvio padrão do F1 externo: referência 3.93%, busca 2.66%


### Configuração final e comparação nas 104 amostras

Para obter uma configuração concreta, rodamos a mesma grade em duas divisões de cinco partes das 416 amostras. A maior média de F1 escolhe a candidata. O melhor resultado dentro dessa busca serve para escolher os parâmetros. A comparação mais útil para estimar desempenho é a das rodadas externas acima.

Só depois dessa escolha aplicamos a candidata e a referência às 104 amostras. Também mostramos o melhor resultado de cada família de distância na busca para localizar os testes. Essa tabela descreve a busca e não é um placar externo.

In [7]:
inicio = perf_counter()
rodadas_busca = np.array([
    executar_divisao(ajuste, consulta)
    for semente in sementes_externas
    for ajuste, consulta in dividir(desenvolvimento, 5, semente)
])
ordem_busca = ordenar_busca(rodadas_busca)
candidata_final = int(ordem_busca[0])
i_final, j_final = ler_configuracao(candidata_final)
m_busca = metricas(rodadas_busca)
f1_busca = m_busca["f1"].mean(axis=0).ravel()
acc_busca = m_busca["acuracia"].mean(axis=0).ravel()

print(f"Tempo da busca nas 416 amostras: {perf_counter() - inicio:.1f} s")
print("Melhor por família na busca:")
mostradas = set()
for posicao in ordem_busca:
    i, j = ler_configuracao(posicao)
    familia = familias[configuracoes[i]]
    if familia in mostradas:
        continue
    mostradas.add(familia)
    print(f"F1 {f1_busca[posicao]:.2%} | Acurácia {acc_busca[posicao]:.2%} | {descrever(posicao)}")
print()
print("Candidata escolhida:", descrever(candidata_final))
print(json.dumps(asdict(configuracoes[i_final]), ensure_ascii=False, indent=2))

Tempo da busca nas 416 amostras: 13.9 s
Melhor por família na busca:
F1 90.38% | Acurácia 90.39% | Combinações planejadas | voto=simples, K=19, limite=45%
F1 89.54% | Acurácia 89.55% | Pesos dos exames | voto=por_posicao, K=25, limite=45%
F1 89.18% | Acurácia 89.18% | Categorias | voto=duplo3, K=41, limite=45%
F1 88.94% | Acurácia 88.95% | Escala | voto=inverso_quadrado, K=41, limite=45%
F1 88.93% | Acurácia 88.94% | Grupos e medidas derivadas | voto=duplo3, K=17, limite=45%
F1 88.82% | Acurácia 88.83% | Retirar uma medida | voto=por_posicao, K=51, limite=45%
F1 88.80% | Acurácia 88.83% | Distância | voto=duplo3, K=17, limite=50%
F1 88.58% | Acurácia 88.58% | Ausências | voto=inverso_quadrado, K=41, limite=45%
F1 88.46% | Acurácia 88.47% | Peso de outra medida | voto=por_posicao, K=31, limite=45%
F1 88.46% | Acurácia 88.46% | Transformação | voto=inverso_quadrado, K=51, limite=45%
F1 88.46% | Acurácia 88.46% | Referência | voto=por_posicao, K=51, limite=45%

Candidata escolhida: Combin

In [8]:
reserva = executar_divisao(desenvolvimento, reservadas, [0, i_final])
referencia_reserva = reserva[0, indice_base]
candidata_reserva = reserva[1, j_final]
print("Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
linha_metricas("Primeira submissão", referencia_reserva)
linha_metricas("Candidata da busca", candidata_reserva)
print()
print("Matrizes [controle, diabetes] — linhas reais, colunas previstas")
print("Primeira submissão:", referencia_reserva.reshape(2, 2).tolist())
print("Candidata da busca:", candidata_reserva.reshape(2, 2).tolist())

Versão                  | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Primeira submissão      |   86.54% |   86.53% |   86.54% |   88.00% |   85.19%
Candidata da busca      |   84.62% |   84.52% |   84.86% |   80.00% |   88.89%

Matrizes [controle, diabetes] — linhas reais, colunas previstas
Primeira submissão: [[44, 6], [8, 46]]
Candidata da busca: [[40, 10], [6, 48]]


### Registro completo dos testes

O arquivo de resultados guarda as quatro contagens da matriz de confusão para cada combinação, em cada rodada da busca e nas divisões internas da avaliação externa. A ordem é TN, FP, FN, TP, com controle como classe 0. As configurações de distância e de voto estão gravadas no mesmo arquivo. Isso permite recuperar qualquer linha da busca sem rodar o notebook novamente.

In [9]:
pasta_resultados = raiz / "resultados"
pasta_resultados.mkdir(exist_ok=True)
caminho_resultados = pasta_resultados / "knn_busca_ampla.npz"
catalogo = {
    "configuracoes": [asdict(c) for c in configuracoes],
    "familias": list(familias.values()),
    "regras": regras,
    "sementes_externas": sementes_externas,
    "eixos_busca": ["rodada", "distancia", "regra_de_voto", "TN_FP_FN_TP"],
    "eixos_internas": ["rodada_externa", "rodada_interna", "distancia", "regra_de_voto", "TN_FP_FN_TP"],
    "eixos_externas": ["rodada_externa", "referencia_ou_busca", "TN_FP_FN_TP"],
    "sha256_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest(),
}
np.savez_compressed(
    caminho_resultados,
    catalogo=json.dumps(catalogo, ensure_ascii=False),
    busca=rodadas_busca,
    internas=np.array(internas),
    externas=externas,
    selecionadas=np.array(selecionadas),
    candidata_final=candidata_final,
    reserva=np.array([referencia_reserva, candidata_reserva]),
)
print("Registro completo:", caminho_resultados.relative_to(raiz))
print("Rodadas da busca:", rodadas_busca.shape[0])
print("Rodadas internas registradas:", len(internas) * 3)
print("Combinações por rodada:", total_configuracoes)

Registro completo: resultados\knn_busca_ampla.npz
Rodadas da busca: 10
Rodadas internas registradas: 30
Combinações por rodada: 54225


#### Como consultar uma combinação salva

Podemos abrir o registro de resultados e consultar uma combinação sem rodar a busca outra vez. A posição começa em zero. O exemplo consulta a primeira colocada e mostra seus parâmetros e a matriz de confusão por rodada. Para ver outra combinação, trocamos a posição informada na chamada.


In [10]:
def consultar_combinacao(posicao, caminho=caminho_resultados):
    with np.load(caminho, allow_pickle=False) as registro:
        dados = json.loads(registro["catalogo"].item())
        quantidade_regras = len(dados["regras"])
        total = len(dados["configuracoes"]) * quantidade_regras
        if not 0 <= posicao < total:
            raise ValueError(f"A posição deve estar entre 0 e {total - 1}.")

        i, j = divmod(posicao, quantidade_regras)
        contagens = registro["busca"][:, i, j]
        voto, k, limite = dados["regras"][j]

    print(f"Posição {posicao} | voto={voto} | K={k} | limite={limite:.0%}")
    print(json.dumps(dados["configuracoes"][i], ensure_ascii=False))
    linha_metricas("Média nas dez rodadas", contagens)
    print("Contagens por rodada [TN, FP, FN, TP]:")
    for numero, matriz in enumerate(contagens, start=1):
        print(f"Rodada {numero:>2}: {matriz.tolist()}")


consultar_combinacao(candidata_final)


Posição 42327 | voto=simples | K=19 | limite=45%
{"escala": "mad", "metrica": "manhattan", "ausencia": "glicose", "transformacao": "nenhuma", "medidas": ["AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM"], "pesos": [["GLUCOSE", 2], ["HEMOGLOBIN", 3]], "categorias": [], "peso_cat": 0.0, "metrica_cat": "jaccard"}
Média nas dez rodadas   |   90.39% |   90.38% |   90.45% |   92.25% |   88.65%
Contagens por rodada [TN, FP, FN, TP]:
Rodada  1: [37, 3, 1, 43]
Rodada  2: [36, 4, 5, 38]
Rodada  3: [35, 5, 7, 36]
Rodada  4: [37, 3, 6, 37]
Rodada  5: [38, 2, 3, 40]
Rodada  6: [35, 5, 8, 36]
Rodada  7: [39, 1, 3, 40]
Rodada  8: [39, 1, 4, 39]
Rodada  9: [36, 4, 7, 36]
Rodada 10: [37, 3, 5, 38]


### Submissão da candidata ao Kaggle

Usamos a primeira colocada da busca, posição 42327, com Manhattan, escala por desvio absoluto mediano, K igual a 19 e votação simples. Pelo menos nove dos 19 vizinhos precisam votar em diabetes. A glicose tem peso 2 e a hemoglobina peso 3. Quando a glicose falta em uma das amostras, ela fica fora da distância. Esta configuração usa só os nove atributos numéricos.

A acurácia média dentro da busca foi 90,39%. Nas 104 amostras reservadas, foi 84,62%. O envio testa essa candidata no conjunto externo da competição.

Para gerar as previsões, calculamos a escala e as medianas com as 520 amostras rotuladas. O teste recebe as mesmas regras numéricas da padronização, incluindo altura em centímetros, separação da pressão e idades 0, -1 e 999 como ausentes. Mantemos todas as linhas e IDs do teste, na ordem original.


In [11]:
def padronizar_numeros_do_teste(amostra):
    entrada = amostra.copy()
    if entrada["AGE"].strip() in {"0", "-1", "999"}:
        entrada["AGE"] = ""

    altura = ler_medida(entrada, "HEIGHT")
    if np.isfinite(altura) and altura < 3:
        entrada["HEIGHT"] = str(altura * 100)

    pressao = entrada["BLOOD_PRESSURE"].strip().replace("x", "/").replace(" ", "")
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"] = ""
        entrada["DIASTOLIC"] = ""
    else:
        sistolica, diastolica = pressao.split("/")
        entrada["SYSTOLIC"] = str(int(sistolica))
        entrada["DIASTOLIC"] = str(int(diastolica))
    return entrada


def matriz_para_novas_amostras(entradas):
    numeros = np.array([[ler_medida(a, coluna) for coluna in numericas] for a in entradas])
    altura_m = numeros[:, indice_num["HEIGHT"]] / 100
    imc = numeros[:, indice_num["WEIGHT"]] / altura_m ** 2
    pulso = numeros[:, indice_num["SYSTOLIC"]] - numeros[:, indice_num["DIASTOLIC"]]
    return np.column_stack((
        numeros, imc, pulso,
        np.isnan(numeros[:, indice_num["GLUCOSE"]]),
        np.isnan(numeros[:, indice_num["HEMOGLOBIN"]]),
    ))


with np.load(raiz / "resultados" / "knn_busca_ampla.npz", allow_pickle=False) as registro:
    posicao_envio = int(registro["candidata_final"].item())
    hash_registrado = json.loads(registro["catalogo"].item())["sha256_base"]
if hashlib.sha256(caminho_base.read_bytes()).hexdigest() != hash_registrado:
    raise ValueError("A base de treinamento mudou desde a busca.")

i_envio, j_envio = ler_configuracao(posicao_envio)
configuracao_envio = configuracoes[i_envio]
if configuracao_envio.categorias or configuracao_envio.peso_cat:
    raise ValueError("Esta preparação de envio foi feita para a candidata só numérica.")

with (raiz / "dados" / "kaggle-test" / "test.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo_teste:
    teste_original = list(csv.DictReader(arquivo_teste))
ids_envio = [a["id"] for a in teste_original]
if not ids_envio or any(not identificador.strip() for identificador in ids_envio):
    raise ValueError("O teste precisa ter IDs preenchidos.")
if len(set(ids_envio)) != len(ids_envio):
    raise ValueError("Os IDs do teste devem ser únicos.")

entradas_envio = [padronizar_numeros_do_teste(a) for a in teste_original]
quantidade_rotulada = len(amostras)
valores = np.vstack((valores[:quantidade_rotulada], matriz_para_novas_amostras(entradas_envio)))
rotulos = np.concatenate((rotulos[:quantidade_rotulada], np.full(len(ids_envio), -1, dtype=np.int8)))
indices_referencia = np.arange(quantidade_rotulada).tolist()
random.Random(42).shuffle(indices_referencia)
indices_referencia = np.array(indices_referencia)
indices_consulta = np.arange(quantidade_rotulada, quantidade_rotulada + len(ids_envio))

classes_vizinhas, distancias_vizinhas = ordenar_configuracao(
    configuracao_envio, indices_referencia, indices_consulta, {}, {}
)
previsoes_envio = previsoes_todas(classes_vizinhas, distancias_vizinhas)[j_envio].astype(int)
linhas_envio = list(zip(ids_envio, previsoes_envio.tolist(), strict=True))
if any(classe not in (0, 1) for _, classe in linhas_envio):
    raise ValueError("Previsões devem pertencer às classes 0 e 1.")

caminho_envio = raiz / "resultados" / "submissao_knn_busca_k19.csv"
with caminho_envio.open("w", encoding="utf-8", newline="") as arquivo_envio:
    escritor = csv.writer(arquivo_envio)
    escritor.writerow(["id", "class"])
    escritor.writerows(linhas_envio)

print(f"Referência com {quantidade_rotulada} amostras rotuladas")
print(f"Arquivo salvo: resultados/{caminho_envio.name}")
print(f"Previsões: {len(linhas_envio)}")
print("Regra de voto:", regras[j_envio])

Referência com 520 amostras rotuladas
Arquivo salvo: resultados/submissao_knn_busca_k19.csv
Previsões: 312
Regra de voto: ('simples', 19, 0.45)
